In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!cp /content/drive/MyDrive/dl-lab-3-product-segmentation.zip /content/


In [5]:
!unzip -q -o /content/dl-lab-3-product-segmentation.zip -d /content/dl-lab-3-product-segmentation/

In [6]:
!ls /content

dl-lab-3-product-segmentation	   drive
dl-lab-3-product-segmentation.zip  sample_data


In [7]:
import torch
print(f"Устройство: {torch.cuda.get_device_name(0)}")
print(f"Памяти доступно: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


Устройство: NVIDIA L4
Памяти доступно: 22.0 GB


In [8]:
!pip install timm
!pip install git+https://github.com/qubvel/segmentation_models.pytorch.git

  Cloning https://github.com/qubvel/segmentation_models.pytorch.git to /tmp/pip-req-build-6hzmn0uj
  Running command git clone --filter=blob:none --quiet https://github.com/qubvel/segmentation_models.pytorch.git /tmp/pip-req-build-6hzmn0uj
  Resolved https://github.com/qubvel/segmentation_models.pytorch.git to commit 99ae07f765d35bd60b79ee248f387b4297be098a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for segmentation_models_pytorch: filename=segmentation_models_pytorch-0.5.1.dev0-py3-none-any.whl size=157855 sha256=2627ef675812e4a0bd9d4a43446eb4f267542440e9d2f27cc3a1f3d51a6ddc04
  Stored in directory: /tmp/pip-ephem-wheel-cache-ezb4sgmx/wheels/c8/6c/a1/5a21a8e8663b96b9308136f28efc4fa6ed1bcac52e7cc5ee27
Successfully built segmentation_models_pytorch


In [10]:
import os
import random
import time
import copy
from pathlib import Path
from tqdm import tqdm
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn
import torchvision.transforms as T
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import KFold

# Проверка CUDA
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")


CUDA available: True
Using GPU: NVIDIA L4
CUDA version: 12.8


In [11]:
import torch
torch.cuda.empty_cache()
import gc
gc.collect()

0

In [13]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from pathlib import Path
from tqdm import tqdm
import time
import copy
from sklearn.model_selection import StratifiedKFold



def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def find_image_for_stem(images_dir: Path, stem: str) -> Path | None:
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def dice_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7, threshold: float = 0.5) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (preds * targets).sum(dim=1)
    denom = preds.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean().item()

def iou_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7, threshold: float = 0.5) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

def mixup_batch(images, masks, alpha=0.2):
    batch_size = images.size(0)
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    index = torch.randperm(batch_size).to(images.device)
    mixed_images = lam * images + (1 - lam) * images[index]
    mixed_masks = lam * masks + (1 - lam) * masks[index]
    return mixed_images, mixed_masks



def get_fg_ratio(mask_path: Path) -> float:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return 0.0
    return float((mask > 0).sum() / mask.size)

def make_area_label(fg_ratio: float) -> int:
    if fg_ratio == 0:
        return 0
    elif fg_ratio < 0.001:
        return 1
    elif fg_ratio < 0.005:
        return 2
    elif fg_ratio < 0.01:
        return 3
    elif fg_ratio < 0.02:
        return 4
    elif fg_ratio < 0.05:
        return 5
    elif fg_ratio < 0.10:
        return 6
    elif fg_ratio < 0.20:
        return 7
    else:
        return 8

def get_samples_with_labels(images_dir: Path, masks_dir: Path):

    samples = []
    labels = []

    mask_files = sorted(masks_dir.glob("*.png"))
    for mask_path in mask_files:
        stem = mask_path.stem
        image_path = find_image_for_stem(images_dir, stem)
        if image_path is not None:
            samples.append((str(image_path), str(mask_path)))
            fg_ratio = get_fg_ratio(mask_path)
            labels.append(make_area_label(fg_ratio))

    return samples, np.array(labels)


class CombinedLoss(nn.Module):
    def __init__(self, dice_weight=0.5, bce_weight=0.2, focal_weight=0.3, label_smoothing=0.05):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True, smooth=0.1)
        self.bce = nn.BCEWithLogitsLoss()
        self.focal = smp.losses.FocalLoss(mode='binary', alpha=0.75, gamma=2.0)
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.focal_weight = focal_weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets_smooth = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        else:
            targets_smooth = targets
        dice_loss = self.dice(logits, targets)
        bce_loss = self.bce(logits, targets_smooth)
        focal_loss = self.focal(logits, targets)
        total_loss = self.dice_weight * dice_loss + self.bce_weight * bce_loss + self.focal_weight * focal_loss
        return total_loss


class TrainTransform:
    def __init__(self, img_size=512):
        self.transform = A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomRotate90(p=0.3),
            A.Affine(scale=(0.9, 1.1), translate_percent=(0.05, 0.05), rotate=(-20, 20), p=0.4),
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
                A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=15, p=0.3),
            ], p=0.5),
            A.GaussNoise(p=0.1),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

    def __call__(self, image, mask):
        augmented = self.transform(image=image, mask=mask)
        return augmented['image'], augmented['mask'].unsqueeze(0).float()

class ValTransform:
    def __init__(self, img_size=512):
        self.transform = A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

    def __call__(self, image, mask):
        augmented = self.transform(image=image, mask=mask)
        return augmented['image'], augmented['mask'].unsqueeze(0).float()



class BinarySegDataset(Dataset):
    def __init__(self, samples, img_size: int = 512, phase: str = 'train'):
        self.samples = samples
        self.img_size = img_size
        self.phase = phase

        if phase == 'train':
            self.transform = TrainTransform(img_size)
        else:
            self.transform = ValTransform(img_size)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        image_path, mask_path = self.samples[idx]
        image_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.float32)
        image, mask = self.transform(image, mask)
        return image, mask



def build_model() -> nn.Module:
    if USE_TIMM_ENCODER:
        try:
            from segmentation_models_pytorch.encoders import TimmUniversalEncoder
            print(f"Using TimmUniversalEncoder with {ENCODER_NAME}")
            encoder = TimmUniversalEncoder(ENCODER_NAME, pretrained=True)
            encoder_name = None
            encoder_weights = None
        except Exception as e:
            print(f"Timm failed: {e}")
            encoder = None
            encoder_name = ENCODER_NAME_SMP
            encoder_weights = ENCODER_WEIGHTS
    else:
        encoder = None
        encoder_name = ENCODER_NAME_SMP
        encoder_weights = ENCODER_WEIGHTS

    model_args = {"in_channels": 3, "classes": 1, "activation": None}

    if encoder is not None:
        model_args["encoder"] = encoder
    else:
        model_args["encoder_name"] = encoder_name
        model_args["encoder_weights"] = encoder_weights

    if MODEL_NAME == "Unet":
        model = smp.Unet(**model_args)
    elif MODEL_NAME == "UnetPlusPlus":
        model = smp.UnetPlusPlus(**model_args)
    elif MODEL_NAME == "FPN":
        model = smp.FPN(**model_args)
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}")

    return model



def train_one_epoch(model, loader, optimizer, loss_fn, device, scaler=None, accum_steps=2):
    model.train()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    pbar = tqdm(loader, desc="Training", unit="batch")
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, (images, masks) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        if USE_MIXUP and np.random.random() > 0.5:
            images, masks = mixup_batch(images, masks, alpha=MIXUP_ALPHA)

        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = loss_fn(logits, masks)
                loss = loss / accum_steps

            scaler.scale(loss).backward()
            if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        else:
            logits = model(images)
            loss = loss_fn(logits, masks) / accum_steps
            loss.backward()
            if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

        actual_loss = loss.item() * accum_steps
        running_loss += actual_loss
        running_dice += dice_score_from_logits(logits.detach(), masks, threshold=THRESHOLD)
        running_iou += iou_score_from_logits(logits.detach(), masks, threshold=THRESHOLD)
        pbar.set_postfix({'loss': f'{running_loss/(batch_idx+1):.4f}', 'dice': f'{running_dice/(batch_idx+1):.4f}'})

    n = len(loader)
    return {"loss": running_loss / n, "dice": running_dice / n, "iou": running_iou / n}

@torch.no_grad()
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()
    running_loss = 0.0
    all_avg_probs = []
    all_masks = []
    scales = [512, 640]

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        batch_probs_list = []
        for s in scales:
            if s != 512:
                img_input = F.interpolate(images, size=(s, s), mode='bilinear', align_corners=False)
            else:
                img_input = images
            p1 = torch.sigmoid(model(img_input))
            p2 = torch.flip(torch.sigmoid(model(torch.flip(img_input, dims=[3]))), dims=[3])
            p3 = torch.flip(torch.sigmoid(model(torch.flip(img_input, dims=[2]))), dims=[2])
            p4 = torch.rot90(torch.sigmoid(model(torch.rot90(img_input, k=1, dims=[2, 3]))), k=-1, dims=[2, 3])
            p_scale = (p1 + p2 + p3 + p4) / 4.0
            if s != 512:
                p_scale = F.interpolate(p_scale, size=(512, 512), mode='bilinear', align_corners=False)
            batch_probs_list.append(p_scale)
        avg_probs = torch.stack(batch_probs_list).mean(dim=0)
        eps = 1e-7
        avg_logits = torch.log(avg_probs + eps) - torch.log(1 - avg_probs + eps)
        loss = loss_fn(avg_logits, masks)
        running_loss += loss.item()
        all_avg_probs.append(avg_probs.cpu())
        all_masks.append(masks.cpu())

    all_avg_probs = torch.cat(all_avg_probs)
    all_masks = torch.cat(all_masks)
    best_dice = 0.0
    best_thr = 0.45
    for thr in [0.35, 0.40, 0.45, 0.50, 0.55]:
        preds = (all_avg_probs > thr).float()
        intersection = (preds * all_masks).sum()
        dice = (2. * intersection + 1e-7) / (preds.sum() + all_masks.sum() + 1e-7)
        if dice > best_dice:
            best_dice = dice.item()
            best_thr = thr
    print(f"Best Validation Threshold: {best_thr} | Dice: {best_dice:.4f}")
    return {"loss": running_loss / len(loader), "dice": best_dice, "iou": best_dice / (2 - best_dice)}


DATA_ROOT = Path("/content/dl-lab-3-product-segmentation")
IMAGES_DIR = DATA_ROOT / "train" / "images"
MASKS_DIR = DATA_ROOT / "train" / "masks"
SAVE_DIR = Path("/content/drive/MyDrive/seg_train_runs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 512
BATCH_SIZE = 12
ACCUM_STEPS = 2
NUM_EPOCHS = 80
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
SEED = 42
THRESHOLD = 0.47

MODEL_NAME = "FPN"
ENCODER_NAME = "efficientnet_b4"
ENCODER_WEIGHTS = "imagenet"

USE_TIMM_ENCODER = True
ENCODER_NAME_SMP = "efficientnet-b4"

USE_SWA = True
SWA_START = 45
SWA_LR = 1e-5
ANNEALING_STRATEGY = "cos"

N_SPLITS = 5
USE_STRATIFIED = True

DICE_WEIGHT = 0.5
BCE_WEIGHT = 0.3
FOCAL_WEIGHT = 0.2
LABEL_SMOOTHING = 0.05

USE_MIXUP = True
MIXUP_ALPHA = 0.2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def main():
    seed_everything(SEED)

    print("="*50)
    print("Step 1: Loading samples...")
    print("="*50)


    all_samples, labels = get_samples_with_labels(IMAGES_DIR, MASKS_DIR)
    print(f"Found {len(all_samples)} paired samples")

    if USE_STRATIFIED:
        print("\nBuilding STRATIFIED folds based on object area...")
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        folds = list(skf.split(all_samples, labels))
        print(f"Created {len(folds)} stratified folds")
    else:
        from sklearn.model_selection import KFold
        kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        folds = list(kf.split(all_samples))

    fold_results = []

    for fold, (train_indices, val_indices) in enumerate(folds):
        print(f"FOLD {fold+1}/{N_SPLITS}")

        train_samples = [all_samples[i] for i in train_indices]
        val_samples = [all_samples[i] for i in val_indices]

        train_dataset = BinarySegDataset(train_samples, IMG_SIZE, phase='train')
        val_dataset = BinarySegDataset(val_samples, IMG_SIZE, phase='val')

        print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
        )
        val_loader = DataLoader(
            val_dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
        )

        print("\n" + "="*50)
        print(f"Step 2 (Fold {fold+1}): Building model...")
        print("="*50)
        model = build_model().to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
        print(f"Using CosineAnnealingLR (T_max={NUM_EPOCHS}, eta_min=1e-6)")

        swa_model = None
        swa_scheduler = None
        if USE_SWA:
            from torch.optim.swa_utils import AveragedModel, SWALR
            swa_model = AveragedModel(model)
            swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR, anneal_strategy=ANNEALING_STRATEGY, anneal_epochs=3)
            print(f"SWA enabled. Will start averaging at epoch {SWA_START}")

        scaler = torch.amp.GradScaler('cuda') if DEVICE == "cuda" else None
        loss_fn = CombinedLoss(
            dice_weight=DICE_WEIGHT, bce_weight=BCE_WEIGHT,
            focal_weight=FOCAL_WEIGHT, label_smoothing=LABEL_SMOOTHING
        )

        best_val_dice = -1.0
        epochs_no_improve = 0

        for epoch in range(1, NUM_EPOCHS + 1):
            print(f"\n{'='*50}")
            print(f"Epoch {epoch}/{NUM_EPOCHS}")
            print(f"{'='*50}")

            epoch_start = time.time()

            train_metrics = train_one_epoch(
                model, train_loader, optimizer, loss_fn, DEVICE, scaler, accum_steps=ACCUM_STEPS
            )

            eval_model = swa_model if (USE_SWA and epoch >= SWA_START) else model
            val_metrics = validate_one_epoch(eval_model, val_loader, loss_fn, DEVICE)

            if not (USE_SWA and epoch >= SWA_START):
                scheduler.step()

            epoch_time = time.time() - epoch_start
            print(f"Epoch {epoch} completed in {epoch_time:.1f}s")
            print(f"  Train - Loss: {train_metrics['loss']:.4f}, Dice: {train_metrics['dice']:.4f}, IoU: {train_metrics['iou']:.4f}")
            print(f"  Val   - Loss: {val_metrics['loss']:.4f}, Dice: {val_metrics['dice']:.4f}, IoU: {val_metrics['iou']:.4f}")

            torch.save({
                "epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(), "val_dice": val_metrics["dice"],
                "config": {"MODEL_NAME": MODEL_NAME, "ENCODER_NAME": ENCODER_NAME, "IMG_SIZE": IMG_SIZE},
            }, SAVE_DIR / f"last_fold_{fold}.pth")

            if val_metrics["dice"] > best_val_dice:
                best_val_dice = val_metrics["dice"]
                state_dict = swa_model.state_dict() if (USE_SWA and epoch >= SWA_START) else model.state_dict()
                epochs_no_improve = 0

                torch.save({
                    "epoch": epoch, "model_state_dict": state_dict, "val_dice": val_metrics["dice"],
                    "config": {"MODEL_NAME": MODEL_NAME, "ENCODER_NAME": ENCODER_NAME, "IMG_SIZE": IMG_SIZE},
                }, SAVE_DIR / f"best_fold_{fold}.pth")
                print(f"✓ Saved new best model for fold {fold} with val_dice={best_val_dice:.4f}")
            else:
                epochs_no_improve += 1
                print(f"No improvement for {epochs_no_improve} epochs.")

            if USE_SWA and epoch >= SWA_START:
                swa_model.update_parameters(model)
                swa_scheduler.step()

            print(f"Current LR: {optimizer.param_groups[0]['lr']:.2e}")

            if epochs_no_improve >= 11:
                print(f"Early Stopping! No improvement for {11} epochs.")
                break

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if USE_SWA and swa_model is not None:
            print("Updating SWA BatchNorm statistics...")
            torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)
            final_val_metrics = validate_one_epoch(swa_model, val_loader, loss_fn, DEVICE)
            print(f"Final SWA Validation Dice: {final_val_metrics['dice']:.4f}")
            if final_val_metrics['dice'] > best_val_dice:
                best_val_dice = final_val_metrics['dice']
                torch.save({
                    "epoch": epoch, "model_state_dict": swa_model.state_dict(), "val_dice": best_val_dice,
                    "config": {"MODEL_NAME": MODEL_NAME, "ENCODER_NAME": ENCODER_NAME, "IMG_SIZE": IMG_SIZE},
                }, SAVE_DIR / f"best_fold_{fold}.pth")
                print(f"✓ SWA model is better! Updated best.pth with Dice: {best_val_dice:.4f}")

        fold_results.append(best_val_dice)
        print(f"\nFold {fold+1} finished. Best Dice: {best_val_dice:.4f}")

    print("\n" + "="*60)
    print(f"K-Fold Cross-Validation Results ({N_SPLITS} folds)")
    print(f"Dice Scores: {[f'{x:.4f}' for x in fold_results]}")
    print(f"Average Dice: {np.mean(fold_results):.4f} +/- {np.std(fold_results):.4f}")
    print("="*60)

if __name__ == "__main__":
    import os
    os.environ['TORCH_HOME'] = '/content/drive/MyDrive/torch_cache'
    main()

Step 1: Loading samples...
Found 2000 paired samples

📊 Building STRATIFIED folds based on object area...
✅ Created 5 stratified folds

🚀 FOLD 1/5
Train size: 1600, Val size: 400

Step 2 (Fold 1): Building model...
✅ Using TimmUniversalEncoder with efficientnet_b4


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


✅ Using CosineAnnealingLR (T_max=80, eta_min=1e-6)
✅ SWA enabled. Will start averaging at epoch 45

Epoch 1/80


Training: 100%|██████████| 134/134 [00:18<00:00,  7.13batch/s, loss=0.3085, dice=0.5809]


🔍 Best Validation Threshold: 0.35 | Dice: 0.8218
Epoch 1 completed in 44.0s
  Train - Loss: 0.3085, Dice: 0.5809, IoU: 0.4684
  Val   - Loss: 0.2339, Dice: 0.8218, IoU: 0.6976
✓ Saved new best model for fold 0 with val_dice=0.8218
Current LR: 1.00e-03

Epoch 2/80


...

🔍 Best Validation Threshold: 0.5 | Dice: 0.9222
Epoch 78 completed in 43.9s
  Train - Loss: 0.1070, Dice: 0.8863, IoU: 0.8209
  Val   - Loss: 0.1160, Dice: 0.9222, IoU: 0.8556
No improvement for 11 epochs.
Current LR: 1.00e-05

Epoch 79/80


Training: 100%|██████████| 134/134 [00:18<00:00,  7.12batch/s, loss=0.1064, dice=0.8892]


🔍 Best Validation Threshold: 0.5 | Dice: 0.9221
Epoch 79 completed in 44.1s
  Train - Loss: 0.1064, Dice: 0.8892, IoU: 0.8235
  Val   - Loss: 0.1164, Dice: 0.9221, IoU: 0.8554
No improvement for 12 epochs.
Current LR: 1.00e-05

Epoch 80/80


Training: 100%|██████████| 134/134 [00:18<00:00,  7.09batch/s, loss=0.1076, dice=0.8871]


🔍 Best Validation Threshold: 0.5 | Dice: 0.9226
Epoch 80 completed in 43.9s
  Train - Loss: 0.1076, Dice: 0.8871, IoU: 0.8200
  Val   - Loss: 0.1152, Dice: 0.9226, IoU: 0.8563
✓ Saved new best model for fold 4 with val_dice=0.9226
Current LR: 1.00e-05
🔄 Updating SWA BatchNorm statistics...
🔍 Best Validation Threshold: 0.5 | Dice: 0.9223
🏁 Final SWA Validation Dice: 0.9223

✅ Fold 5 finished. Best Dice: 0.9226

🏆 K-Fold Cross-Validation Results (5 folds)
Dice Scores: ['0.9310', '0.9166', '0.9291', '0.9274', '0.9226']
Average Dice: 0.9253 +/- 0.0052


In [ ]:
import json
import torch
import cv2
import numpy as np
import pandas as pd
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from pathlib import Path
from tqdm import tqdm
from collections import OrderedDict

TEST_IMAGES_DIR = Path("/content/dl-lab-3-product-segmentation/test_images")
OUTPUT_CSV = "submission_ensemble_5folds.csv"
CHECKPOINTS_DIR = Path("/content/drive/MyDrive/seg_train_runs")

FOLD_CHECKPOINTS = [
    CHECKPOINTS_DIR / "best_fold_0.pth",
    CHECKPOINTS_DIR / "best_fold_1.pth",
    CHECKPOINTS_DIR / "best_fold_2.pth",
    CHECKPOINTS_DIR / "best_fold_3.pth",
    CHECKPOINTS_DIR / "best_fold_4.pth",
]

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

THRESHOLD = 0.47
MIN_AREA = 150
SCALES = [512, 640]

CLOSE_KERNEL = 5
OPEN_KERNEL = 3

ENSEMBLE_METHOD = "mean"


def cv2_imread_unicode(path: Path, flags=cv2.IMREAD_COLOR):
    data = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(data, flags) if data.size > 0 else None

def fix_state_dict(state_dict):
    """Убираем префикс 'module.' и ключи SWA"""
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        if k == "n_averaged":
            continue

        if k.startswith('module.'):
            name = k[7:]
        else:
            name = k
        new_state_dict[name] = v
    return new_state_dict

def build_model_with_timm_encoder(model_name, encoder_name):
    from segmentation_models_pytorch.encoders import TimmUniversalEncoder

    print(f"   Building model with TimmUniversalEncoder: {encoder_name}")

    encoder = TimmUniversalEncoder(encoder_name, pretrained=False)

    if model_name == "Unet":
        model = smp.Unet(encoder=encoder, in_channels=3, classes=1, activation=None)
    elif model_name == "UnetPlusPlus":
        model = smp.UnetPlusPlus(encoder=encoder, in_channels=3, classes=1, activation=None)
    elif model_name == "FPN":
        model = smp.FPN(encoder=encoder, in_channels=3, classes=1, activation=None)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model

def preprocess_image_timm(image_rgb, size):
    import timm
    data_config = timm.data.resolve_data_config({}, model_name=ENCODER_NAME)
    transforms = timm.data.create_transform(**data_config, is_training=False)

    image = cv2.resize(image_rgb, (size, size), interpolation=cv2.INTER_LINEAR)
    image = transforms(image).unsqueeze(0)
    return image

def preprocess_image_simple(image_rgb, size):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    image = cv2.resize(image_rgb, (size, size), interpolation=cv2.INTER_LINEAR)
    image = image.astype(np.float32) / 255.0
    image = (image - mean) / std
    image = torch.from_numpy(image.transpose(2, 0, 1)).float().unsqueeze(0)
    return image

def serialize_mask(mask2d: np.ndarray) -> str:
    return json.dumps(mask2d.astype(np.uint8).tolist(), separators=(",", ":"))

def postprocess_mask(mask: np.ndarray, min_area: int = 150) -> np.ndarray:
    if mask.sum() == 0:
        return mask

    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (CLOSE_KERNEL, CLOSE_KERNEL))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (OPEN_KERNEL, OPEN_KERNEL))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    for j in range(1, num_labels):
        if stats[j, cv2.CC_STAT_AREA] < min_area:
            mask[labels == j] = 0

    return mask

@torch.no_grad()
def predict_single_model(model, img_rgb, base_size, device):
    msi_probs = []

    for s in SCALES:
        img_tensor = preprocess_image_simple(img_rgb, s).to(device)

        p1 = torch.sigmoid(model(img_tensor))
        p2 = torch.flip(torch.sigmoid(model(torch.flip(img_tensor, dims=[3]))), dims=[3])
        p3 = torch.flip(torch.sigmoid(model(torch.flip(img_tensor, dims=[2]))), dims=[2])
        img_90 = torch.rot90(img_tensor, k=1, dims=[2, 3])
        p4 = torch.rot90(torch.sigmoid(model(img_90)), k=-1, dims=[2, 3])

        p_scale = (p1 + p2 + p3 + p4) / 4.0

        if s != base_size:
            p_scale = F.interpolate(p_scale, size=(base_size, base_size), mode='bilinear', align_corners=False)

        msi_probs.append(p_scale)

    return torch.stack(msi_probs).mean(dim=0)[0, 0]

@torch.no_grad()
def predict_ensemble(models, img_rgb, base_size, device, method="mean"):
    all_probs = []

    for model in models:
        prob = predict_single_model(model, img_rgb, base_size, device)
        all_probs.append(prob.cpu().numpy())

    all_probs = np.stack(all_probs, axis=0)

    if method == "mean":
        return np.mean(all_probs, axis=0)
    elif method == "median":
        return np.median(all_probs, axis=0)
    else:
        return np.mean(all_probs, axis=0)



print("ЗАГРУЗКА АНСАМБЛЯ ИЗ 5 ФОЛДОВ (TimmUniversalEncoder)")


models = []
val_dices = []
base_size = None
encoder_name = None

for i, ckpt_path in enumerate(FOLD_CHECKPOINTS):
    if not ckpt_path.exists():
        print(f"Чекпоинт {ckpt_path} не найден!")
        continue

    print(f"\nЗагрузка {ckpt_path.name}...")
    checkpoint = torch.load(ckpt_path, map_location="cpu")
    cfg = checkpoint["config"]

    if base_size is None:
        base_size = cfg["IMG_SIZE"]
    if encoder_name is None:
        encoder_name = cfg["ENCODER_NAME"]

    print(f"   Config: {cfg['MODEL_NAME']} + {encoder_name}")

    model = build_model_with_timm_encoder(cfg["MODEL_NAME"], encoder_name)

    state_dict = fix_state_dict(checkpoint["model_state_dict"])

    model.load_state_dict(state_dict, strict=True)
    model.to(DEVICE).eval()
    models.append(model)

    val_dice = checkpoint.get("val_dice", 0.0)
    val_dices.append(val_dice)

    print(f"Fold {i} загружен | Val Dice: {val_dice:.4f}")

print(f"\n{'='*60}")
print(f"Загружено моделей: {len(models)}/{len(FOLD_CHECKPOINTS)}")
if val_dices:
    print(f" Val Dice: {np.mean(val_dices):.4f} ± {np.std(val_dices):.4f}")
print(f"Метод ансамблирования: {ENSEMBLE_METHOD}")
print("="*60)


image_paths = sorted([p for p in TEST_IMAGES_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS])
print(f"Найдено {len(image_paths)} тестовых изображений")

rows = []
failed_images = []

for img_path in tqdm(image_paths, desc="Инференс (ансамбль 5 фолдов)"):
    img_bgr = cv2_imread_unicode(img_path)
    if img_bgr is None:
        failed_images.append(img_path.name)
        continue

    H, W = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    pred_prob = predict_ensemble(models, img_rgb, base_size, DEVICE, method=ENSEMBLE_METHOD)

    if pred_prob.shape != (H, W):
        pred_prob = cv2.resize(pred_prob, (W, H), interpolation=cv2.INTER_LINEAR)

    mask = (pred_prob > THRESHOLD).astype(np.uint8)
    mask = postprocess_mask(mask, min_area=MIN_AREA)

    rows.append({"ImageId": img_path.name, "mask": serialize_mask(mask)})


df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print(f"Результаты сохранены в {OUTPUT_CSV}")


ЗАГРУЗКА АНСАМБЛЯ ИЗ 5 ФОЛДОВ (TimmUniversalEncoder)

Загрузка best_fold_0.pth...
   Config: FPN + efficientnet_b4
   Building model with TimmUniversalEncoder: efficientnet_b4
Fold 0 загружен | Val Dice: 0.9310

Загрузка best_fold_1.pth...
   Config: FPN + efficientnet_b4
   Building model with TimmUniversalEncoder: efficientnet_b4
Fold 1 загружен | Val Dice: 0.9166

Загрузка best_fold_2.pth...
   Config: FPN + efficientnet_b4
   Building model with TimmUniversalEncoder: efficientnet_b4
Fold 2 загружен | Val Dice: 0.9291

Загрузка best_fold_3.pth...
   Config: FPN + efficientnet_b4
   Building model with TimmUniversalEncoder: efficientnet_b4
Fold 3 загружен | Val Dice: 0.9274

Загрузка best_fold_4.pth...
   Config: FPN + efficientnet_b4
   Building model with TimmUniversalEncoder: efficientnet_b4
Fold 4 загружен | Val Dice: 0.9226

Загружено моделей: 5/5
 Val Dice: 0.9253 ± 0.0052
Метод ансамблирования: mean
Найдено 2000 тестовых изображений


Инференс (ансамбль 5 фолдов):  21%|██        | 417/2000 [02:43<10:21,  2.55it/s]

In [ ]:
!cp submission_ensemble_5folds_final.csv /content/drive/MyDrive/


In [ ]:
from google.colab import runtime
runtime.unassign()

